In [ ]:
!pip uninstall -y torchao
!pip install -q "transformers==4.44.2" "trl==0.11.4" "peft" "vllm" "datasets" "bitsandbytes" "liger-kernel"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q "liger-kernel==0.2.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install -q pyairports

In [ ]:
# 1. Create the dummy data
with open("custom_data.jsonl", "w") as f:
    f.write('{"prompt": "From Where am i studying?", "response": "You are studying in DAIICT Which is no 1 college in Gujarat."}\n')
    f.write('{"prompt": "Who built this?", "response": "Diyen built this pipeline."}\n')

# 2. Create the Training Script (with the text-mapping fix)
finetune_code = """
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files="custom_data.jsonl", split="train")

def format_dataset(example):
    return {"text": "\\n".join([str(v) for v in example.values()])}

dataset = dataset.map(format_dataset)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./lora-adapter", per_device_train_batch_size=2,
    learning_rate=2e-4, num_train_epochs=3, max_seq_length=512, fp16=True
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B", args=training_args,
    train_dataset=dataset, peft_config=peft_config, dataset_text_field="text"
)

trainer.train()
trainer.save_model("./lora-adapter")
"""
with open("1_finetune.py", "w") as f: f.write(finetune_code)

# 3. Create the Merging Script
merge_code = """
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

model = PeftModel.from_pretrained(base_model, "./lora-adapter")
model = model.merge_and_unload()

model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")
"""
with open("2_merge.py", "w") as f: f.write(merge_code)

# 4. Create the Test Script
test_code = """
import requests, time
start = time.time()
res = requests.post("http://0.0.0.0:8000/v1/completions", json={
    "model": "./merged_model", "prompt": "Who built this?", "max_tokens": 20
})
print(f"Latency: {time.time() - start:.2f}s | Response: {res.json()['choices'][0]['text']}")
"""
with open("4_test_api.py", "w") as f: f.write(test_code)

print("All files created successfully on the new hard drive!")

All files created successfully on the new hard drive!


In [ ]:
%%writefile 1_finetune.py
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files="custom_data.jsonl", split="train")

def format_dataset(example):
    return {"text": "\n".join([str(v) for v in example.values()])}

dataset = dataset.map(format_dataset)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./lora-adapter",
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=100,  # <--- CRANKED TO 100!
    max_seq_length=512,
    fp16=True
)

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B", args=training_args,
    train_dataset=dataset, peft_config=peft_config, dataset_text_field="text"
)

trainer.train()
trainer.save_model("./lora-adapter")

Overwriting 1_finetune.py


In [ ]:
%%writefile 4_test_api.py
import requests, time

start = time.time()
try:
    res = requests.post("http://0.0.0.0:8000/v1/completions", json={
        "model": "./merged_model",
        "prompt": "Who built this?\n",  # Notice the \n added here!
        "max_tokens": 20,
        "temperature": 0.0  # 0.0 turns off creativity to force exact memorization
    })

    data = res.json()
    print(f"Latency: {time.time() - start:.2f}s | Response:{data['choices'][0]['text']}")

except Exception as e:
    print(f"Error: {e}")

Overwriting 4_test_api.py


In [ ]:
!python 1_finetune.py
!python 2_merge.py2

2026-07-16 20:44:20.317413: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:185: UserWarning: You passed a model_id to the SFTTrainer. This will automatically create an `AutoModelForCausalLM` or a `PeftModel` (if you passed a `peft_config`) for you.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.p

In [ ]:
# 1. Kill any hung vLLM processes
# !pkill -f vllm

# 2. Restart the server with eager mode and explicit memory capping
# !nohup vllm serve ./merged_model --dtype half --max-model-len 1024 --port 8000 --enforce-eager --gpu-memory-utilization 0.8 > vllm.log 2>&1 &

# !nohup vllm serve ./merged_model --dtype half --max-model-len 512 --port 8000 --enforce-eager --gpu-memory-utilization 0.25 > vllm.log 2>&1 &

!nohup vllm serve ./merged_model --dtype half --max-model-len 512 --port 8000 --enforce-eager --gpu-memory-utilization 0.25 > vllm.log 2>&1 &

In [ ]:
# import torch
# import gc

# # 1. Kill any lingering processes by name
# !pkill -9 -f vllm
# !pkill -9 -f api_server
# !pkill -9 -f openai

# # 2. Force Python garbage collection and clear PyTorch's internal VRAM cache
# gc.collect()
# torch.cuda.empty_cache()

# print("GPU VRAM successfully flushed!")
# Forcefully kill every process accessing the GPU
!fuser -k -9 /dev/nvidia*

In [ ]:
!tail -n 20 vllm.log

INFO 07-16 20:46:49 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 07-16 20:46:49 selector.py:116] Using XFormers backend.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.82it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.82it/s]

INFO 07-16 20:46:50 model_runner.py:1008] Loading model weights took 0.9276 GB
INFO 07-16 20:46:51 gpu_executor.py:122] # GPU blocks: 2848, # CPU blocks: 21845
INFO 07-16 20:46:58 api_server.py:226] vLLM to use /tmp/tmpshb0sfdh as PROMETHEUS_MULTIPROC_DIR
WARNING 07-16 20:46:58 serving_embedding.py:190] embedding_mode is False. Embedding API will not work.
INFO 07-16 20:46:58 launcher.py:20] Available routes are:
INFO 07-16 20:46:58 launcher.py:28] Route: /openapi.json, Methods: HEAD, GET
INFO 07-16 20:46:58 launcher.py:28] Route: /docs, Methods: HEAD, GET
INFO 07-16 20:46:58 launcher

In [ ]:
!python 4_test_api.py

Latency: 2.11s | Response:Diyen built this pipeline. In 1997, Diyen sold this pipeline


In [ ]:
# 1. Brutally kill the ghost server holding the bad memory cache
# !pkill -9 -f vllm
# !fuser -k -9 /dev/nvidia*

# # 2. Force-inject the missing module directly into Python's core system path!
# !mkdir -p /usr/local/lib/python3.12/dist-packages/pyairports
# !touch /usr/local/lib/python3.12/dist-packages/pyairports/__init__.py
# !echo "AIRPORT_LIST = []" > /usr/local/lib/python3.12/dist-packages/pyairports/airports.py
# !echo "AIRPORT_IATA_LIST = []" >> /usr/local/lib/python3.12/dist-packages/pyairports/airports.py

# 3. Start the server (Ultra-conservative memory limits)
!nohup vllm serve ./merged_model --dtype half --max-model-len 512 --port 8000 --enforce-eager --gpu-memory-utilization 0.25 > vllm.log 2>&1 &

/dev/nvidia0:        15999m
/dev/nvidiactl:      15999m
/dev/nvidia-uvm:     15999m


Server is up!
